# ORQUESTACION - PIPELINE ANALISIS DE COBRANZAS
## Preparación de entorno

In [ ]:
# LIBRERIAS DE TRABAJO: Descomente y ejecute la siguiente línea de ser necesario
#%pip install xlrd sqlparse PyMySQL SQLAlchemy requests pandas mysql-connector-python cryptography 

In [2]:
import pandas as pd
import requests
import os
import io
import unicodedata #para manejo de tildes
#-------------------------------------- 
from sqlalchemy import create_engine, text
from sqlalchemy.types import String
import getpass #para manejo de password seguro en conexion a Mysql
import sqlparse #diseñada para entender la anatomía de un archivo SQL, ignorar comentarios y separar las sentencias correctamente.
import mysql.connector
import pymysql
from pymysql.constants import CLIENT

In [3]:
# Conexión a repo GitHub
usuario = "Marcelo-F-Martin"
repo = "PI_UA_pipeline_analisis_de_cobranza"

# URL de la API para acceder al último release donde se alojan los archivos en crudo.
api_url = f"https://api.github.com/repos/{usuario}/{repo}/releases/latest"

respuesta = requests.get(api_url)
archivos_crudos = respuesta.json().get('assets', []) # El segundo argumento [] del .get(), es para que no rompa el codigo si la llave 'assets' no existe.

## Inicio proceso ETL
### 1- Extracción:
#####   Obtención de datos crudos desde la fuente y almacenamiento en Dataframe de Pandas

In [4]:
df_definitivo_por_libro = []   
total_registros = 0
#====== INICIO bucle externo: recorre archivos dentro del directorio ===============================
for archivo in archivos_crudos:
    
    # Filtrado para leer solo archivos .xls
    if archivo['name'].endswith('.xls'):
        print(f"Nombre de archivo encontrado: {archivo['name']}")
        
    # lista para incorporar los df por cada hoja de cada libro.
    hojas_consolidadas_por_libro = [] 
    
    url_archivo = archivo['browser_download_url']
    respuesta_url_archivo = requests.get(url_archivo).content
    archivo = pd.read_excel(io.BytesIO(respuesta_url_archivo), engine='xlrd', sheet_name=None, header=None)

    #====== INICIO 1º bucle interno: recorre hojas del archivo =======================================
    for nombre_hoja, df_crudo in archivo.items():
                   
        # Aquí 'df_bruto' es el DataFrame bruto de la hoja actual
        print(f"✅ Hoja '{nombre_hoja}' cargada correctamente. Filas brutas: {len(df_crudo)}")
       
        # Retornamos el diccionario para el siguiente paso del procesamiento
        # print(f"  --- Buscando clave en Hoja '{nombre_hoja}' ---")
        
        encabezado_clave = "Periodo"
        indice_encabezado_fila = df_crudo[df_crudo.apply(lambda row: encabezado_clave in row.astype(str).values, axis=1)].index
        
        fila_inicio = indice_encabezado_fila[0]
        print(f"✅ Encabezado clave '{encabezado_clave}' encontrado en la Fila (índice 0): {fila_inicio}")

        fila_encabezado = df_crudo.iloc[fila_inicio]

        # 2b. 'col_inicio' será el índice de la columna donde se encuentra la clave
        # Esto es crucial para ignorar las columnas añadidas a la izquierda.
        
        col_inicio = fila_encabezado[fila_encabezado.astype(str) == encabezado_clave].index[0]
        #print(f"  ✅ Columna de inicio de datos (índice 0): {col_inicio}")

        df_crudo.columns = df_crudo.iloc[fila_inicio] # Asignar la fila encontrada como nuevo encabezado
        df_limpio = df_crudo[fila_inicio + 1:].reset_index(drop=True) # Eliminar filas superiores

        df_final_hoja = df_limpio.copy()
        df_final_hoja['hoja_origen'] = nombre_hoja

        
        
        if not df_final_hoja.empty:
            hojas_consolidadas_por_libro.append(df_final_hoja)
            print(f"  ✅ Hoja '{nombre_hoja}' añadida a la consolidación. Registros: {len(df_final_hoja)}")
            total_registros += len(df_final_hoja)
        else:
            print(f"  Advertencia: Hoja '{nombre_hoja}' vacía después de la limpieza. No se añadió.")
       
    df_archivo_unificado = pd.concat(hojas_consolidadas_por_libro, ignore_index=True)
    df_definitivo_por_libro.append(df_archivo_unificado)
            
    #====== FIN bucleinterno =================================================
#====== FIN bucle externo ====================================================

# Unificación final
df_final = pd.concat(df_definitivo_por_libro, ignore_index=True)
print(f' Total de registros: {total_registros}')

Nombre de archivo encontrado: Detalle.de.comisiones.Broker_2.xls
✅ Hoja 'B0001' cargada correctamente. Filas brutas: 33008
✅ Encabezado clave 'Periodo' encontrado en la Fila (índice 0): 7
  ✅ Hoja 'B0001' añadida a la consolidación. Registros: 33000
Nombre de archivo encontrado: Detalle.de.comisiones.Carlos.Diaz_1.xls
✅ Hoja 'AI002' cargada correctamente. Filas brutas: 787
✅ Encabezado clave 'Periodo' encontrado en la Fila (índice 0): 7
  ✅ Hoja 'AI002' añadida a la consolidación. Registros: 779
✅ Hoja 'AI003' cargada correctamente. Filas brutas: 754
✅ Encabezado clave 'Periodo' encontrado en la Fila (índice 0): 7
  ✅ Hoja 'AI003' añadida a la consolidación. Registros: 746
✅ Hoja 'AI013' cargada correctamente. Filas brutas: 640
✅ Encabezado clave 'Periodo' encontrado en la Fila (índice 0): 7
  ✅ Hoja 'AI013' añadida a la consolidación. Registros: 632
Nombre de archivo encontrado: Detalle.de.comisiones.Juan.Perez_1.xls
✅ Hoja 'AI001' cargada correctamente. Filas brutas: 3607
✅ Encabezad

### 2- Exploración y Transformación

In [ ]:
df_final.info()

In [8]:
def limpiar_tilde(texto):
    # 1. Normaliza el texto a la forma NFD (Descomposición)
    # Esto separa la 'á' en 'a' + 'tilde combinable'
    texto_normalizado = unicodedata.normalize('NFD', texto)
    
    # 2. Filtra y mantiene solo los caracteres que no sean "marcas" de acento (Mn)
    texto_sin_acentos = "".join(
        c for c in texto_normalizado if unicodedata.category(c) != 'Mn'
    )
   
    return texto_sin_acentos

In [9]:
# 1_estandariza a minusculas
# 2_elimina espacios en blanco
# 3_reemplaza vacíos por '_'
# 4_aplica función definida para limpiar tildes
#________________________________________________

df_final.columns = df_final.columns.str.lower().str.strip().str.replace(' ','_')
df_final.columns = [limpiar_tilde(col) for col in df_final.columns]
df_final.columns

Index(['periodo', 'region', 'asesor', 'denominacion_asesor', 'broker',
       'id_broker', 'fecha_operacion', 'contrato', 'rama', 'especialidad',
       'id_cliente', 'nombre_cliente', 'comision', 'valor_neto',
       'porcentaje_comision', 'forma_pago', 'numero_recibo', 'hoja_origen'],
      dtype='str')

In [10]:
# seleccion de columnas relevantes a importar a la Base de Datos
lista_columnas_seleccionadas = ['periodo','asesor','fecha_operacion','contrato','comision', 'valor_neto', 'porcentaje_comision','forma_pago', 'numero_recibo', 'hoja_origen'  ]
df_final = df_final[lista_columnas_seleccionadas]

In [ ]:
df_final.head()

In [ ]:
# Verifica existencia de PK
df_final.nunique()

In [11]:
df_final_limpio = df_final.copy()
df_final_limpio

,periodo,asesor,fecha_operacion,contrato,comision,valor_neto,porcentaje_comision,forma_pago,numero_recibo,hoja_origen
0,202506,AE047,2025-06-11 00:00:00,GUPW9,1730.97,34619.33,0.05,efe,24807,B0001
1,202510,AE006,2025-10-20 00:00:00,JN4TM,1906.91,38138.17,0.05,transf,78856,B0001
2,202511,AE006,2025-11-23 00:00:00,WKB45,1558.11,31162.18,0.05,transf,71496,B0001
3,202503,AE011,2025-03-29 00:00:00,55T20,1673.15,16731.54,0.1,efe,81111,B0001
4,202508,AE026,2025-08-09 00:00:00,VK0Q0,1598.25,15982.49,0.1,transf,38615,B0001
...,...,...,...,...,...,...,...,...,...,...
45995,202510,AI004,2025-10-26 00:00:00,RVUES,3357.106,16785.53,0.2,tc,18246,AI004
45996,202501,AI004,2025-01-03 00:00:00,3FKC9,706.393,7063.93,0.1,transf,63092,AI004
45997,202503,AI004,2025-03-02 00:00:00,H75FR,8387.84,41939.2,0.2,transf,86040,AI004
45998,202508,AI004,2025-08-20 00:00:00,RYSTM,3505.787,35057.87,0.1,efe,77539,AI004


In [ ]:
# Esta celda puede ser opcional para exportar el dataframe en CSV.

#ruta_archivo = r"C:\Users\orglo\OneDrive\Desktop\marcelo\Proyectos\PI"
#nombre_csv_salida = f"comisiones_consolidadas.csv"
#ruta_salida_completa = os.path.join(ruta_archivo, nombre_csv_salida)
#df_final.to_csv(ruta_salida_completa, index=False, encoding='utf-8')


### 3- Carga
#### Conexión a MySQL

In [ ]:
# Verifique los datos de su entorno en MySQL
usuario = "root"
host = "localhost"
puerto = "3306"
bd_capa_uno = "cobranzas_capa_uno"
bd_capa_dos = "cobranzas_capa_dos"
tabla_temp = "temp_cobranzas"

# Acceso a scripts SQL en repositorio
url_sql_ddl = 'https://raw.githubusercontent.com/Marcelo-F-Martin/PI_UA_pipeline_analisis_de_cobranza/refs/heads/main/SQL/1_PI_UA_DDL.sql'
url_sql_insert_fact = 'https://raw.githubusercontent.com/Marcelo-F-Martin/PI_UA_pipeline_analisis_de_cobranza/refs/heads/main/SQL/2_PI_UA_insert_en_fact.sql'
url_sql_vistas = 'https://raw.githubusercontent.com/Marcelo-F-Martin/PI_UA_pipeline_analisis_de_cobranza/refs/heads/main/SQL/4_PI_UA_capa_dos_vistas.sql'
url_sql_sp = 'https://raw.githubusercontent.com/Marcelo-F-Martin/PI_UA_pipeline_analisis_de_cobranza/refs/heads/main/SQL/3_PI_UA_SP_calendario.sql'


#### Ejecución de scripts SQL
##### 1- crea base de datos y tablas | 2- inserta datos en tabla temporal | 3- inserta datos en tabla fact | 4- crea SP tabla_calendario | 5- crea vistas en BD capa_dos

In [ ]:
try:
    # Descargar el contenido del archivo
    respuesta_1 = requests.get(url_sql_ddl)
    respuesta_2 = requests.get(url_sql_insert_fact)    
    respuesta_3 = requests.get(url_sql_vistas)  
    respuesta_4 = requests.get(url_sql_sp)  
    
    if respuesta_1.status_code == 200 and respuesta_2.status_code == 200 and respuesta_3.status_code == 200 and respuesta_4.status_code == 200:
        script_sql_1 = respuesta_1.text
        print("✅ Script SQL DDL descargado correctamente desde repo GitHub.")

        script_sql_2 = respuesta_2.text
        print("✅ Script SQL Insert en 'fact_crobranzas' descargado correctamente desde repo GitHub.")

        script_sql_3 = respuesta_3.text
        print("✅ Script SQL Create Views descargado correctamente desde repo GitHub.")

        script_sql_4 = respuesta_4.text
        print("✅ Script SQL Create SP descargado correctamente desde repo GitHub.")
        
        # Manejo seguro de credenciales de BD.
        password = getpass.getpass("Ingrese su contraseña de MySQL: ")    
        
        engine_create = create_engine(f"mysql+pymysql://{usuario}:{password}@{host}:{puerto}") # crea nuevas BD.
        engine_insert = create_engine(f"mysql+pymysql://{usuario}:{password}@{host}:{puerto}/{bd_capa_uno}") # inserta datos en tabla temp_cobranzas.
        engine_vista = create_engine(f"mysql+pymysql://{usuario}:{password}@{host}:{puerto}/{bd_capa_dos}") # crea vistas

        # 1.DDL crea base de datos y tablas        
        with engine_create.begin() as connection:

            for statement in script_sql_1.split(';'):
                if statement.strip():
                    connection.execute(text(statement))

        # 2.Ingesta el dataframe en tabla temporal (truncate + insert)
        with engine_insert.begin() as connection:
            connection.execute(text(f"TRUNCATE TABLE {tabla_temp}"))
            print(f"🧹 Tabla '{tabla_temp}' limpiada (registros eliminados).")
            
            df_final_limpio.to_sql(
                                    name=tabla_temp,
                                    con=connection,
                                    if_exists='append', 
                                    index=False,
                                    chunksize=1000, 
                                    method='multi' # Para que sea masivo y no registro por registro
                                  )
           
        print(f"✅ Ingesta exitosa: {len(df_final_limpio)} registros insertados.")

        # 3.Pasa datos de tabla_temp a tabla_fact        
        with engine_insert.begin() as connection:
 
            for statement in script_sql_2.split(';'):
                if statement.strip():
                    connection.execute(text(statement))
        
        print("🚀 Scripts a tabla Fact en MySQL finalizada exitosamente.")


        # 4.Crea Stored Procedure en BD cobranzas_capa_uno 
        conn = pymysql.connect(
                                host=host,
                                user=usuario,
                                password=password,
                                db=bd_capa_uno,
                                port=int(puerto),
                                client_flag=CLIENT.MULTI_STATEMENTS
                               )     

        with conn.cursor() as cursor:
             cursor.execute(script_sql_4)
        
        conn.commit()
        conn.close()

        print("🚀 Script SP en MySQL finalizada exitosamente.")        
        
        # 5.Crea vistas en BD cobranzas_capa_dos        
        with engine_vista.begin() as connection:
             
            for statement in script_sql_3.split(';'):
                if statement.strip():
                    connection.execute(text(statement))
        
        print("🚀 Script Vistas en MySQL finalizada exitosamente.")
        
    else:
        print(f"❌ Error al acceder al archivo Nº 1: Estado {respuesta_1.status_code}")
        print(f"❌ Error al acceder al archivo Nº 2: Estado {respuesta_2.status_code}")
        print(f"❌ Error al acceder al archivo Nº 3: Estado {respuesta_3.status_code}")
        print(f"❌ Error al acceder al archivo Nº 4: Estado {respuesta_4.status_code}")

except Exception as e:
    print(f"❌ Error durante el proceso: {e}")
print(f"===================================================")